# Verso Fine-Tuning — Qwen 2.5 1.5B with Unsloth

Fine-tunes Qwen 2.5 1.5B to generate learning reels + flashcards as JSON.

**Google Colab setup:** Use a T4 GPU runtime (free tier works!)
- Runtime → Change runtime type → T4 GPU

**Steps:**
1. Install Unsloth (Colab-compatible)
2. Upload training data (`verso_training_sharegpt_v2.json`)
3. Load model (4-bit quantized)
4. Apply LoRA adapters
5. Train on ShareGPT v2 dataset
6. Export GGUF for Ollama

In [ ]:
%%capture
# Cell 1: Install Unsloth (Colab-compatible)
# Do NOT use --no-deps alone, it skips unsloth_zoo
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
# Cell 2: Upload your training data
# Option A: Upload from your machine via Colab's file picker
# Option B: Upload to Google Drive and mount

import os, json

# --- Option A: Direct upload ---
from google.colab import files

DATA_PATH = "verso_training_sharegpt_v2.json"
if not os.path.exists(DATA_PATH):
    print("Upload verso_training_sharegpt_v2.json now:")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

with open(DATA_PATH) as f:
    data = json.load(f)
print(f"Loaded {len(data)} training examples")
print(f"Sample keys: {list(data[0].keys())}")

In [ ]:
# Cell 3: Load Qwen 2.5 1.5B Instruct with 4-bit quantization
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048  # reduced from 4096, saves VRAM on T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (fp16 on T4)
    load_in_4bit=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Cell 4: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # memory efficient
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")

In [ ]:
# Cell 5: Prepare dataset in ShareGPT/chat format
from datasets import Dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Load and standardize
dataset = Dataset.from_list(data)
dataset = standardize_sharegpt(dataset)

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)
print(f"Dataset ready: {len(dataset)} examples")
print(f"\nSample (first 500 chars):\n{dataset[0]['text'][:500]}")

In [ ]:
# Cell 6: Configure trainer (T4/Colab-optimized)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,       # reduced from 4 for T4
        gradient_accumulation_steps=8,       # effective batch size = 16
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),    # auto-detect: fp16 on T4, bf16 on A100
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./verso_output",
        report_to="none",
    ),
)

print("Trainer configured. Ready to train.")

In [ ]:
# Cell 7: Train!
import time

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final loss: {stats.training_loss:.4f}")

In [ ]:
# Cell 8: Quick test — generate a sample output
FastLanguageModel.for_inference(model)

test_prompt = """Document type: textbook
Learning style: visual
Depth: balanced
Focus: learning
Difficulty: medium

Text:
Photosynthesis is the process by which green plants convert light energy into chemical energy. It occurs in chloroplasts and involves two stages: light reactions and the Calvin cycle.

JSON:"""

messages = [
    {"role": "system", "content": "You are Verso, a learning content creator who teaches through short reels.\nYou are NOT a textbook. You explain like a friend.\n\nCRITICAL RULES YOU MUST FOLLOW:\n1. You MUST use at least 3 contractions (don't, isn't, you're, it's, here's) in every narration.\n2. You MUST use \"...\" at least once and \"\u2014\" at least once in every narration.\n3. You must NEVER use these phrases: \"is defined as\", \"refers to the process\", \"plays a crucial role\", \"it is important to note\", \"furthermore\", \"moreover\".\n4. Narration MUST be 40-60 words. Count carefully.\n5. Always output valid JSON with \"reels\" and \"flashcards\" arrays."},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
output = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.3)
response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

print("=== Test Output ===")
print(response[:1000])

In [ ]:
# Cell 9: Export to GGUF for Ollama
model.save_pretrained_gguf(
    "verso-qwen2.5-1.5b",
    tokenizer,
    quantization_method="q4_k_m",
)

print("\nGGUF export complete!")

# Find and show the GGUF file
import glob
gguf_files = glob.glob("verso-qwen2.5-1.5b/*.gguf") + glob.glob("verso-qwen2.5-1.5b-unsloth.Q4_K_M.gguf")
for f in gguf_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"Found: {f} ({size_mb:.0f} MB)")

# Download to your machine from Colab
from google.colab import files
for f in gguf_files:
    print(f"\nDownloading {f}...")
    files.download(f)

## Next Steps

1. The `.gguf` file will auto-download to your machine after Cell 9
2. Place it in `scripts/verso-qwen2.5-1.5b_gguf/` directory
3. Create the Ollama model:
   ```bash
   cd scripts
   ollama create verso-reel-v2 -f Modelfile.v2
   ```
4. A/B evaluate against the base model:
   ```bash
   cd backend && python ../scripts/ab_eval_models.py --model-a qwen2.5:1.5b --model-b verso-reel-v2
   ```
5. If the gate passes, switch to the fine-tuned model:
   ```bash
   export REEL_MODEL=verso-reel-v2
   ```

## Troubleshooting
- **Kernel crash / OOM**: Runtime → Restart runtime, then re-run from Cell 1
- **Slow training**: Free Colab T4 is ~2x slower than RTX 3090, expect ~15-20 min for 3 epochs with 1.5B
- **Disconnects**: Enable "Prevent disconnection" browser extensions for long training runs